# Convert Excel ke CSV

In [5]:
from pathlib import Path
import pandas as pd

input_dir = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\excel")
output_dir = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv")

excel_extensions = {".xlsx"}
excel_files = [
    file for file in input_dir.iterdir()
    if file.is_file() and file.suffix.lower() in excel_extensions
]

if not excel_files:
    raise FileNotFoundError(f"Tidak ada file Excel di: {input_dir}")

for excel_file in excel_files:
    data = pd.read_excel(excel_file)  # Membaca sheet pertama
    csv_file = output_dir / f"{excel_file.stem}.csv"
    data.to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"Berhasil: {excel_file.name} -> {csv_file.name}")

Berhasil: Kultivasi 1_Final Data.xlsx -> Kultivasi 1_Final Data.csv
Berhasil: Kultivasi 2_Final Data.xlsx -> Kultivasi 2_Final Data.csv
Berhasil: Kultivasi 3_Final Data.xlsx -> Kultivasi 3_Final Data.csv
Berhasil: Kultivasi 4_Final Data.xlsx -> Kultivasi 4_Final Data.csv
Berhasil: Kultivasi 5_Final Data.xlsx -> Kultivasi 5_Final Data.csv
Berhasil: Kultivasi 6_Final Data.xlsx -> Kultivasi 6_Final Data.csv


# MERGE FILE CSV WITH SAME COLUMN

In [8]:
from pathlib import Path
import pandas as pd

csv_dir = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv")
output_file = csv_dir / "gabungan.csv"

kolom_pilihan = [
    "Hari",
    "Tanggal",
    "Waktu (jam:menit:detik)",
    "Optical density (AU) (680 nm).Lab",
    "Biomassa (g/L)",
]

csv_files = [
    file for file in csv_dir.glob("*.csv")
    if file != output_file
]

if not csv_files:
    raise FileNotFoundError(f"Tidak ada file CSV di folder: {csv_dir}")

dataframes = []

for file in csv_files:
    df = pd.read_csv(file, encoding="utf-8-sig", usecols=kolom_pilihan)
    # Memastikan urutan kolom di setiap file mengikuti daftar pilihan
    dataframes.append(df[kolom_pilihan])

hasil = pd.concat(dataframes, ignore_index=True)
hasil.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Berhasil menggabungkan {len(csv_files)} file.")
print(f"Kolom yang disimpan: {kolom_pilihan}")
print(f"File hasil: {output_file}")

Berhasil menggabungkan 6 file.
Kolom yang disimpan: ['Hari', 'Tanggal', 'Waktu (jam:menit:detik)', 'Optical density (AU) (680 nm).Lab', 'Biomassa (g/L)']
File hasil: D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\gabungan.csv


# FIX FORMAT

In [10]:
from pathlib import Path
from datetime import datetime
import re
import pandas as pd

csv_file = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\gabungan.csv")

df = pd.read_csv(csv_file, encoding="utf-8-sig", dtype=str, keep_default_na=False)

bulan_id = {
    "januari": "January",
    "februari": "February",
    "maret": "March",
    "april": "April",
    "mei": "May",
    "juni": "June",
    "juli": "July",
    "agustus": "August",
    "september": "September",
    "oktober": "October",
    "november": "November",
    "desember": "December",
}

def rapikan_tanggal(nilai):
    teks = nilai.strip()
    if not teks:
        return ""

    # Ubah nama bulan Indonesia menjadi bahasa Inggris agar bisa dibaca strptime.
    for nama_bulan, nama_inggris in bulan_id.items():
        teks = re.sub(nama_bulan, nama_inggris, teks, flags=re.IGNORECASE)

    for format_tanggal in (
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S",
        "%d %B %Y",
        "%d %b %Y",
        "%d/%m/%Y",
        "%d-%m-%Y",
    ):
        try:
            return datetime.strptime(teks, format_tanggal).strftime("%Y-%m-%d")
        except ValueError:
            continue

    raise ValueError(f"Format tanggal tidak dikenali: {nilai!r}")

def rapikan_waktu(nilai):
    teks = nilai.strip()
    if not teks:
        return ""

    # Mengambil jam dari nilai seperti "1900-01-01 10:04:48".
    cocok = re.search(r"(?<!\d)(\d{1,2}:\d{2}(?::\d{2})?)(?!\d)", teks)
    if not cocok:
        raise ValueError(f"Format waktu tidak dikenali: {nilai!r}")

    teks_jam = cocok.group(1)
    for format_waktu in ("%H:%M:%S", "%H:%M"):
        try:
            return datetime.strptime(teks_jam, format_waktu).strftime("%H:%M:%S")
        except ValueError:
            continue

    raise ValueError(f"Format waktu tidak dikenali: {nilai!r}")

df["Tanggal"] = df["Tanggal"].map(rapikan_tanggal)
df["Waktu (jam:menit:detik)"] = df["Waktu (jam:menit:detik)"].map(rapikan_waktu)

# Ubah koma desimal menjadi titik pada kolom-kolom ini, jika ada.
kolom_angka = ["Biomassa (g/L)", "Optical density (AU) (680 nm).Lab"]

for kolom in kolom_angka:
    if kolom in df.columns:
        df[kolom] = pd.to_numeric(
            df[kolom].str.replace(",", ".", regex=False).replace("", pd.NA),
            errors="raise",
        )

# Menyimpan hasil kembali ke gabungan.csv
df.to_csv(csv_file, index=False, encoding="utf-8-sig")

print(f"Berhasil merapikan file: {csv_file}")

Berhasil merapikan file: D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\gabungan.csv
